# Notebook 03 — TF.Data Input Pipeline

1. Build `tf.data.Dataset` from split CSVs
2. Apply augmentations (train only)
3. Compute class weights
4. Visualize sample batches

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('.'))

from utils.tfdata_utils import build_dataset, get_class_weights

print(f'TensorFlow: {tf.__version__}')
print(f'GPUs: {len(tf.config.list_physical_devices("GPU"))}')

In [ ]:
# ── Configuration ────────────────────────────────────────────────
DATA_DIR   = 'processed_data'
BATCH_SIZE = 16
IMG_SIZE   = (224, 224)

STAGE_NAMES = ['Stage I', 'Stage II', 'Stage III', 'Stage IV']

In [ ]:
# ── Load split info ──────────────────────────────────────────────
train_csv = os.path.join(DATA_DIR, 'train_slices.csv')
val_csv   = os.path.join(DATA_DIR, 'val_slices.csv')
test_csv  = os.path.join(DATA_DIR, 'test_slices.csv')

for name, csv in [('Train', train_csv), ('Val', val_csv), ('Test', test_csv)]:
    df = pd.read_csv(csv)
    print(f'{name}: {len(df)} slices, labels: {dict(df["label"].value_counts().sort_index())}')

In [ ]:
# ── Build datasets ───────────────────────────────────────────────
train_ds = build_dataset(
    csv_path=train_csv, base_dir=DATA_DIR,
    batch_size=BATCH_SIZE, augment=True, shuffle=True,
    target_size=IMG_SIZE
)

val_ds = build_dataset(
    csv_path=val_csv, base_dir=DATA_DIR,
    batch_size=BATCH_SIZE, augment=False, shuffle=False,
    target_size=IMG_SIZE
)

test_ds = build_dataset(
    csv_path=test_csv, base_dir=DATA_DIR,
    batch_size=BATCH_SIZE, augment=False, shuffle=False,
    target_size=IMG_SIZE
)

print('Datasets built successfully')
for name, ds in [('Train', train_ds), ('Val', val_ds), ('Test', test_ds)]:
    for batch in ds.take(1):
        imgs, labels = batch
        print(f'  {name}: batch shape={imgs.shape}, labels shape={labels.shape}')

In [ ]:
# ── Class weights ────────────────────────────────────────────────
train_labels = pd.read_csv(train_csv)['label'].values
class_weights = get_class_weights(train_labels)
print('Class weights:')
for cls, weight in class_weights.items():
    name = STAGE_NAMES[cls] if cls < len(STAGE_NAMES) else f'Class {cls}'
    print(f'  {name} (class {cls}): {weight:.4f}')

In [ ]:
# ── Visualize a training batch (with augmentation) ────────────────
for images, labels in train_ds.take(1):
    fig, axes = plt.subplots(2, 8, figsize=(20, 5))
    for i in range(min(16, images.shape[0])):
        ax = axes[i // 8, i % 8]
        ax.imshow(images[i, :, :, 0].numpy(), cmap='gray')
        label = int(labels[i].numpy())
        name = STAGE_NAMES[label] if label < len(STAGE_NAMES) else f'C{label}'
        ax.set_title(name, fontsize=9)
        ax.axis('off')
    plt.suptitle('Training Batch (augmented)', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Visualize augmentation effect on single image ────────────────
from utils.tfdata_utils import _augment

for images, labels in val_ds.take(1):
    original = images[0]
    fig, axes = plt.subplots(1, 6, figsize=(18, 3))
    axes[0].imshow(original[:, :, 0].numpy(), cmap='gray')
    axes[0].set_title('Original')
    axes[0].axis('off')

    for i in range(1, 6):
        aug_img, _ = _augment(original, labels[0])
        axes[i].imshow(aug_img[:, :, 0].numpy(), cmap='gray')
        axes[i].set_title(f'Aug {i}')
        axes[i].axis('off')

    plt.suptitle('Augmentation Variations', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Save class weights for notebook 04 ───────────────────────────
import json

weights_path = os.path.join(DATA_DIR, 'class_weights.json')
# Convert keys to strings for JSON
with open(weights_path, 'w') as f:
    json.dump({str(k): v for k, v in class_weights.items()}, f, indent=2)
print(f'Saved class weights to {weights_path}')

print('\n✅ tf.data pipeline ready for training')